# Week 4 Lab: Error Diagnosis — Slices, Calibration, and Bug Hunting

**ECBS5200 — Practical Deep Learning Engineering**

In Week 3 you trained two models and made a deployment recommendation.
Today's question: **how do you know you were right?**

Aggregate accuracy and macro F1 hide almost everything interesting.
A model can be 57% accurate and catastrophically broken on the
complaints you care about most. A 2-point macro F1 gap between two
models can be real, or it can be noise.

This lab gives you the diagnostic toolkit:
- **Slice analysis** — where does each model underperform?
- **Calibration** — can you trust the model's confidence?
- **Confusion-matrix reading** — when the model is wrong, where does
  it go wrong?
- **Val-set reliability** — how much should you trust a macro F1 number?
- **Bug hunting** — given a broken model, diagnose it from symptoms alone.

**Time budget:** ~80 minutes.

**How to use this notebook:**
- **GUIDED** cells run as-is. Read them.
- **INTERACTIVE** cells require you to fill in values or write answers.
- `___` is a placeholder that will cause a NameError if not filled in.
- Some sections ask you to **PREDICT** before observing — write down
  your prediction first, then reveal.

## Kaggle setup (do this first!)

1. **Persistence** → "Variables and Files"
2. **Internet** → On
3. **Accelerator** → GPU T4

In [ ]:
import subprocess, sys, os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# HuggingFace config — set BEFORE importing HF libraries
if os.path.exists('/kaggle/working'):
    os.environ['HF_HOME'] = '/kaggle/working/.hf_cache'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '60'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "transformers>=4.53", "datasets", "accelerate", "scikit-learn",
    "matplotlib", "pandas", "tqdm", "peft",
])
print("Packages installed.")

from huggingface_hub import login
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if hf_token:
    login(token=hf_token)
    print("Logged in to HuggingFace.")
else:
    print("WARNING: No HF_TOKEN found. May hit rate limits.")

In [ ]:
import gc
import re
import time
import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

# Load course utilities
REPO_PATH = Path("../..").resolve()
if not (REPO_PATH / "utils" / "data_utils.py").exists():
    REPO_PATH = Path("/tmp/course")
    if not REPO_PATH.exists():
        subprocess.check_call(["git", "clone", "--depth", "1",
            "https://github.com/earino/applied-deep-learning.git", str(REPO_PATH)])
sys.path.insert(0, str(REPO_PATH))

from utils.data_utils import (
    load_course_data, LABEL_LIST, NUM_LABELS, LABEL_TO_ID, ID_TO_LABEL, MAX_LENGTH,
)

In [ ]:
# Reproducibility and device
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability()
    USE_BF16 = cc[0] >= 8
    USE_FP16 = not USE_BF16
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Precision: {'bf16' if USE_BF16 else 'fp16'}")
else:
    USE_BF16, USE_FP16 = False, False
    print("No GPU detected.")

---
## Part 1: Load Models and Run Inference (~10 min)

You trained an encoder and a decoder in Week 3. The merged checkpoints
are on HuggingFace Hub. Load both and run them on the val set — the
diagnostic analyses in the rest of the lab all work from these predictions.
---

In [ ]:
# GUIDED: Load the canonical dataset
print("Loading dataset...")
train_ds, val_ds, _ = load_course_data()
print(f"Train: {len(train_ds):,}  Val: {len(val_ds):,}  Classes: {NUM_LABELS}")

In [ ]:
# GUIDED: HF Hub checkpoints from Week 3 (merged — load directly, no PEFT needed)
ENCODER_REPO = "earino/ecbs5200-week3-encoder-lora"
DECODER_REPO = "earino/ecbs5200-week3-decoder-lora"

In [ ]:
# GUIDED: Shared helper — run inference and return a dict of arrays
def run_inference(model, tokenizer, dataset):
    """Run batched inference over the val set; return logits, preds, labels.
    Tokenizer's padding_side must already be set correctly by the caller."""
    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True,
                         max_length=MAX_LENGTH, padding=False)

    ds_tok = dataset.map(tokenize, batched=True, remove_columns=["text", "label_name"])
    ds_tok = ds_tok.rename_column("label", "labels")

    collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True,
                                       pad_to_multiple_of=8)
    trainer = Trainer(
        model=model,
        args=TrainingArguments(output_dir="/tmp/eval", per_device_eval_batch_size=64,
                               report_to="none"),
        data_collator=collator,
    )
    out = trainer.predict(ds_tok)
    logits = out.predictions
    labels = out.label_ids
    probs = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()
    preds = probs.argmax(-1)
    max_conf = probs.max(-1)
    return {
        "logits": logits.astype(np.float32),
        "probs": probs.astype(np.float32),
        "preds": preds.astype(np.int64),
        "labels": labels.astype(np.int64),
        "max_conf": max_conf.astype(np.float32),
    }

In [ ]:
# GUIDED: Load encoder + run inference
print(f"Loading encoder from {ENCODER_REPO}...")
enc_tokenizer = AutoTokenizer.from_pretrained(ENCODER_REPO)
enc_model = AutoModelForSequenceClassification.from_pretrained(
    ENCODER_REPO, attn_implementation="sdpa",
).to(device).eval()

print("Running inference on encoder...")
t0 = time.time()
enc = run_inference(enc_model, enc_tokenizer, val_ds)
print(f"Done in {time.time() - t0:.1f}s")
print(f"  Accuracy: {(enc['preds'] == enc['labels']).mean():.4f}")
print(f"  Macro F1: {f1_score(enc['labels'], enc['preds'], average='macro', zero_division=0):.4f}")

del enc_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# GUIDED: Load decoder + run inference (LEFT padding — you saw this in Week 3)
print(f"\nLoading decoder from {DECODER_REPO}...")
dec_tokenizer = AutoTokenizer.from_pretrained(DECODER_REPO)
dec_tokenizer.padding_side = "left"
if dec_tokenizer.pad_token is None:
    dec_tokenizer.pad_token = dec_tokenizer.eos_token

dec_model = AutoModelForSequenceClassification.from_pretrained(DECODER_REPO).to(device).eval()
if dec_model.config.pad_token_id is None:
    dec_model.config.pad_token_id = dec_tokenizer.pad_token_id

print("Running inference on decoder...")
t0 = time.time()
dec = run_inference(dec_model, dec_tokenizer, val_ds)
print(f"Done in {time.time() - t0:.1f}s")
print(f"  Accuracy: {(dec['preds'] == dec['labels']).mean():.4f}")
print(f"  Macro F1: {f1_score(dec['labels'], dec['preds'], average='macro', zero_division=0):.4f}")

del dec_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Both models live at around 57% accuracy and macro F1 between 0.21 and 0.24.
From Week 3 you already know the decoder has the aggregate edge.

**But the aggregate numbers are a summary.** Where does each model win?
Where does it lose? When you say "the decoder is better," better at what,
on which complaints? That's what the rest of the lab is for.

---
## Part 2: Slice Analysis (~15 min)

A slice is a subset of the val set defined by some property of the input
— text length, presence of redacted content, all-caps density, etc.
If a model's performance varies meaningfully across slices, that's where
the aggregate number is hiding information.

We'll look at two slice axes in the lab. The homework has four more.
---

### Slice axis 1: character length

In [ ]:
# GUIDED: Compute character length for every val example
val_char_length = np.array([len(t) for t in val_ds["text"]])
char_q = np.percentile(val_char_length, [25, 50, 75])
print(f"Char length quartile cuts: {char_q}")

def char_bucket(c):
    if c <= char_q[0]: return "q1_shortest"
    if c <= char_q[1]: return "q2"
    if c <= char_q[2]: return "q3"
    return "q4_longest"

char_slice = np.array([char_bucket(c) for c in val_char_length])
print(f"Slice sizes: {dict(Counter(char_slice))}")

In [ ]:
# INTERACTIVE: For each slice, compute the macro F1 of both models
# Hint: mask the preds/labels arrays by slice, then call f1_score.

print(f"{'slice':<14} {'n':>5} {'encoder F1':>12} {'decoder F1':>12} {'diff':>7}")
print("-" * 56)
for slice_name in ["q1_shortest", "q2", "q3", "q4_longest"]:
    mask = char_slice == slice_name
    enc_slice_f1 = f1_score(enc["labels"][mask], ___, average="macro", zero_division=0)
    dec_slice_f1 = f1_score(___, dec["preds"][mask], average="macro", zero_division=0)
    diff = dec_slice_f1 - enc_slice_f1
    print(f"{slice_name:<14} {mask.sum():>5} {enc_slice_f1:>12.4f} "
          f"{dec_slice_f1:>12.4f} {diff:>+7.4f}")

### INTERACTIVE: Read the table

Look at the encoder-vs-decoder gap across the four quartiles. Three of
the four quartiles show the decoder winning by a similar amount. One
quartile is different.

**Which quartile is the outlier, and what do you think is special about
that slice of complaints?**

Write your answer before you reveal.

YOUR OBSERVATION:


<details>
<summary><b>Click after writing your take</b></summary>

The q3 quartile (medium-longer complaints, roughly 300-400 chars) is
where the encoder matches the decoder — the gap essentially vanishes.
The decoder's advantage is biggest on the shortest and longest quartiles.

A plausible read: q3 is the "sweet spot" for the encoder's 128-token
window — enough context to make a decision, not so much that important
content gets truncated. On very short complaints the decoder's richer
pretraining knowledge helps (less signal in the complaint itself, more
weight on priors). On very long complaints, truncation kicks in and
the decoder is better at squeezing signal out of the first 128 tokens.

The takeaway isn't this specific story — it's that **a single "the
decoder is better" aggregate conclusion collapses a per-slice picture
where the truth is more nuanced.**

</details>

### Slice axis 2: redaction (presence of "XXXX")

The CFPB data is redacted — account numbers, names, dates, and amounts
are replaced with "XXXX" before publication. Some complaints are heavily
redacted ("on XXXX I called XXXX and spoke to XXXX"), some aren't.
Does redaction-heaviness affect either model?

In [ ]:
# GUIDED: Compute redaction flag
XXXX_RE = re.compile(r"X{4}")
redact_slice = np.array(["redacted" if XXXX_RE.search(t) else "clean"
                         for t in val_ds["text"]])
print(f"Redacted: {(redact_slice == 'redacted').sum()}  "
      f"Clean: {(redact_slice == 'clean').sum()}")

In [ ]:
# INTERACTIVE: Per-slice F1 for redaction
print(f"{'slice':<12} {'n':>5} {'encoder F1':>12} {'decoder F1':>12} {'diff':>7}")
print("-" * 54)
for slice_name in ["clean", "redacted"]:
    mask = redact_slice == slice_name
    enc_slice_f1 = ___  # Macro F1 of encoder on this slice
    dec_slice_f1 = ___  # Macro F1 of decoder on this slice
    diff = dec_slice_f1 - enc_slice_f1
    print(f"{slice_name:<12} {mask.sum():>5} {enc_slice_f1:>12.4f} "
          f"{dec_slice_f1:>12.4f} {diff:>+7.4f}")

### INTERACTIVE: Read the redaction table

Which slice has the bigger encoder-vs-decoder gap, and why might that be?

YOUR OBSERVATION:


<details>
<summary><b>Click after writing your take</b></summary>

The gap is ~4× bigger on **redacted** complaints than on clean ones.
Clean complaints: +0.009 (basically tied). Redacted: +0.041.

One read: the decoder was pretrained on ~18T tokens of heterogeneous
web text and is more robust to "noisy" input (XXXX tokens, broken
syntax around them). The encoder was pretrained on cleaner text and
loses more signal when XXXX patterns disrupt its representations.

For deployment: if your production traffic is heavily redacted, the
encoder-vs-decoder choice isn't an equal tradeoff — the decoder's
advantage is concentrated exactly in the regime your product cares about.

</details>

---
## Part 3: Calibration (~15 min)

Pre-work module 5 introduced ECE (Expected Calibration Error)
abstractly. This is the week you measure it on models you trained.

**The question calibration answers:** when a model says "I'm 80%
confident in this prediction," is it right 80% of the time?
A well-calibrated model's confidence matches its accuracy. A
miscalibrated one is either overconfident (says 90%, right 70%) or
underconfident (says 60%, right 80%).
---

### INTERACTIVE: Predict before you measure

You have two models. One is a 149M-parameter encoder pretrained on
~2T English tokens. The other is a 494M-parameter decoder pretrained
on ~18T multilingual tokens. Both were fine-tuned with LoRA for the
same 3 epochs on the same data.

**PREDICTION:** Which model will have lower ECE (better calibration)?
Write your guess AND your reasoning — folk wisdom about model size
and calibration may or may not hold here.

YOUR PREDICTION:



In [ ]:
# GUIDED: ECE computation
def compute_ece(confidences, correct, n_bins=15):
    """Expected Calibration Error. Weighted mean gap between confidence
    and accuracy across equal-width bins."""
    bins = np.linspace(0, 1, n_bins + 1)
    bin_ids = np.clip(np.digitize(confidences, bins) - 1, 0, n_bins - 1)
    ece = 0.0
    rows = []
    for b in range(n_bins):
        mask = bin_ids == b
        n = mask.sum()
        if n == 0:
            rows.append({"lo": bins[b], "hi": bins[b+1], "n": 0,
                         "mean_conf": 0.0, "accuracy": 0.0})
            continue
        mean_conf = confidences[mask].mean()
        acc = correct[mask].mean()
        ece += (n / len(confidences)) * abs(mean_conf - acc)
        rows.append({"lo": float(bins[b]), "hi": float(bins[b+1]), "n": int(n),
                     "mean_conf": float(mean_conf), "accuracy": float(acc)})
    return float(ece), rows

In [ ]:
# INTERACTIVE: Compute ECE for both models
# Hint: the "correct" array is 1 where preds == labels, 0 otherwise.

enc_correct = (enc["preds"] == enc["labels"]).astype(np.float32)
dec_correct = ___   # same shape as above, for the decoder

enc_ece, enc_bins = compute_ece(enc["max_conf"], enc_correct, n_bins=15)
dec_ece, dec_bins = compute_ece(___, ___, n_bins=15)

print(f"Encoder ECE: {enc_ece:.4f}")
print(f"Decoder ECE: {dec_ece:.4f}")

### The reveal

**Was your prediction right?**

YOUR OBSERVATION:


<details>
<summary><b>Click after writing your take</b></summary>

Folk wisdom would say: the decoder is bigger, saw more pretraining
data, so it should be better calibrated. On this task, the opposite is
true — the decoder's ECE (~0.10) is substantially worse than the
encoder's (~0.06). The decoder is **more overconfident**.

Why? Pretraining scale doesn't guarantee calibration. Calibration is
downstream of the fine-tuning signal — how the model's output
distribution gets shaped by the cross-entropy loss you trained it with.
On long-tail classification with 113 classes and sparse data, any
fine-tuned classifier tends to drift toward overconfidence, and larger
models drift further because they have more expressive power to
concentrate probability mass.

The lesson: **never predict a model's calibration from its size.
Measure it.**

</details>

In [ ]:
# GUIDED: Reliability diagram
fig, ax = plt.subplots(figsize=(7, 6))
for name, bins, ece, color in [("encoder", enc_bins, enc_ece, "#3498db"),
                               ("decoder", dec_bins, dec_ece, "#e74c3c")]:
    xs = [b["mean_conf"] for b in bins if b["n"] > 0]
    ys = [b["accuracy"]  for b in bins if b["n"] > 0]
    ax.plot(xs, ys, marker="o", color=color,
            label=f"{name} (ECE={ece:.3f})", linewidth=1.5, markersize=7)
ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="perfect calibration")
ax.set_xlabel("mean predicted confidence")
ax.set_ylabel("empirical accuracy")
ax.set_title("Reliability diagram — encoder vs decoder")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

**Reading the reliability diagram:** for each confidence bin, we plot
(mean confidence, empirical accuracy). Perfect calibration is the
diagonal — at 80% confidence, 80% of predictions are right.

A line ABOVE the diagonal = underconfident (model is more accurate
than it claims).
A line BELOW the diagonal = overconfident (model is less accurate
than it claims). The decoder should sit below the diagonal for most bins.

Temperature scaling is a one-parameter fix for this. You'll do that
in the homework — fit a scalar T on half of val, apply `softmax(logits / T)`
to the other half, and measure how much ECE drops.

---
## Part 4: Confusion Matrix — Where Does Error Go? (~20 min)

Pre-work module 4 showed you a 5×5 confusion matrix. With 25 cells it
was easy to eyeball. Today you have **113×113** — 12,769 cells, most
of them small.

At that scale, the interesting questions aren't about individual cells.
They're about structural patterns. Here's the structural question this
section answers: **when a rare-class complaint is misclassified, where
does the wrong prediction land?**
---

### Tier definitions

We'll bucket the 113 classes by training frequency into three tiers:
- **Head** = top 20 by training frequency (the most common 20 classes)
- **Mid**  = next 40 classes
- **Tail** = bottom 53 classes

In [ ]:
# GUIDED: Build the tier sets
train_label_counts = Counter(train_ds["label"])
classes_by_freq = [lid for lid, _ in train_label_counts.most_common()]

HEAD_CLASSES = set(classes_by_freq[:20])
MID_CLASSES  = set(classes_by_freq[20:60])
TAIL_CLASSES = set(classes_by_freq[60:])

print(f"Head: {len(HEAD_CLASSES)} classes, "
      f"train-freq range {max(train_label_counts[c] for c in HEAD_CLASSES)} → "
      f"{min(train_label_counts[c] for c in HEAD_CLASSES)}")
print(f"Mid:  {len(MID_CLASSES)} classes, "
      f"range {max(train_label_counts[c] for c in MID_CLASSES)} → "
      f"{min(train_label_counts[c] for c in MID_CLASSES)}")
print(f"Tail: {len(TAIL_CLASSES)} classes, "
      f"range {max(train_label_counts[c] for c in TAIL_CLASSES)} → "
      f"{min(train_label_counts[c] for c in TAIL_CLASSES)}")

### INTERACTIVE: Predict before you measure

Consider val examples whose true label is a **tail** class (one of the
bottom 53 by training frequency). Most of them will be misclassified
— macro F1 on these is low. The interesting question is WHERE the
wrong predictions land.

There are three possible destinations:
- **Head class** (the model defaults to one of the top 20)
- **Mid class**
- **Other tail class** (confused with a different rare class)

**PREDICTION:** When a tail-class example is misclassified, what
fraction of those wrong predictions go to OTHER TAIL classes? Write
a number between 0 and 100.

Most people say something like "a lot — similar rare classes get
confused with each other." You might say differently. Write your guess.

YOUR PREDICTION (percentage):


In [ ]:
# GUIDED: Tier breakdown helper
def tier_breakdown_tail(preds, labels):
    """For val examples where y_true is a tail class AND the prediction is
    wrong, count where the predictions land: head / mid / tail-other."""
    tail_mask = np.isin(labels, list(TAIL_CLASSES))
    wrong_mask = preds != labels
    err_mask = tail_mask & wrong_mask
    err_preds = preds[err_mask]

    to_head = sum(p in HEAD_CLASSES for p in err_preds)
    to_mid  = sum(p in MID_CLASSES for p in err_preds)
    to_tail = sum(p in TAIL_CLASSES for p in err_preds)
    n = len(err_preds)
    return {
        "n_tail_examples": int(tail_mask.sum()),
        "n_tail_errors":   int(n),
        "pct_to_head":     100 * to_head / n if n else 0.0,
        "pct_to_mid":      100 * to_mid / n if n else 0.0,
        "pct_to_tail":     100 * to_tail / n if n else 0.0,
    }

In [ ]:
# GUIDED: Compute for the encoder
br_enc = tier_breakdown_tail(enc["preds"], enc["labels"])
print("ENCODER — tail-class error distribution")
print(f"  Tail examples: {br_enc['n_tail_examples']}    "
      f"Tail errors: {br_enc['n_tail_errors']}")
print(f"  tail → head:         {br_enc['pct_to_head']:5.1f}%")
print(f"  tail → mid:          {br_enc['pct_to_mid']:5.1f}%")
print(f"  tail → other tail:   {br_enc['pct_to_tail']:5.1f}%   <-- the number you predicted")

### The reveal

Compare the "other tail" number to your prediction. Was it higher or
lower?

Most people predict 30-60% for tail→tail because rare classes "should"
get confused with each other. The actual number is much lower.

<details>
<summary><b>Click after comparing your prediction</b></summary>

Only ~10% of tail-class errors go to other tail classes. ~90% of
tail errors **escape the tail entirely** — predictions land in head
or mid classes.

This reshapes what "class imbalance" means operationally. The intuition
that rare classes get "out-shouted by the majority" or "confused with
similar neighbors" — neither is really what's happening. Rare-class
errors diffuse across the entire non-tail label space roughly evenly
between head and mid.

**Practical consequence:** you can't fix rare-class performance just
by weighting the majority class down. That's solving the wrong problem.
The rare-class examples aren't drowning in majority-class noise —
they're being confused with a broad swath of larger-class semantic
neighbors. Different fix, different scope, different cost.

</details>

In [ ]:
# GUIDED: Run the same analysis on the decoder
br_dec = tier_breakdown_tail(dec["preds"], dec["labels"])
print("DECODER — tail-class error distribution")
print(f"  Tail examples: {br_dec['n_tail_examples']}    "
      f"Tail errors: {br_dec['n_tail_errors']}")
print(f"  tail → head:         {br_dec['pct_to_head']:5.1f}%")
print(f"  tail → mid:          {br_dec['pct_to_mid']:5.1f}%")
print(f"  tail → other tail:   {br_dec['pct_to_tail']:5.1f}%")

### INTERACTIVE: Same pattern on the decoder?

The encoder and decoder are very different architectures with different
pretraining data. Does the 90/10 split hold on both?

What does the answer tell you about whether this is an ARCHITECTURE
property or a PROPERTY OF THE DATA?

YOUR OBSERVATION:



### Callback to today's lecture

Before you move on: some of the errors you just bucketed into head/mid
aren't the model's fault. From the opening pipeline module: `"Advertising"`
is still **five separate classes** in the canonical 113, and
`"Struggling to pay your bill"` was **merged into** `"Struggling to pay
your loan"`. Some mid-tier confusions are ours, not the model's.

You'll drill into specific confusion pairs in the homework (item 4).
When you do, flag at least one pair that looks **pipeline-induced**
rather than model-induced.

---
## Part 5: Val-Set Reliability (~5 min)

Before you trust a macro F1 number, ask: how many val examples does
each class have? F1 on a class with n=1 val example is a coin flip.
Noise floor matters.
---

In [ ]:
# GUIDED: Val examples per class
val_label_counts = Counter(val_ds["label"])
val_counts = np.array([val_label_counts.get(i, 0) for i in range(NUM_LABELS)])

In [ ]:
# INTERACTIVE: Histogram — how many classes fall in each bucket?
# Fill in the bucket logic.

buckets = {
    "n=0":     ___,   # count of classes with val count 0
    "n=1":     ___,   # count with val count exactly 1
    "n=2":     int((val_counts == 2).sum()),
    "n=3-4":   int(((val_counts >= 3) & (val_counts <= 4)).sum()),
    "n=5-10":  int(((val_counts >= 5) & (val_counts <= 10)).sum()),
    "n=11-50": int(((val_counts >= 11) & (val_counts <= 50)).sum()),
    "n=51+":   int((val_counts > 50).sum()),
}

print("Val examples per class — histogram:")
for bucket, count in buckets.items():
    bar = "█" * count
    print(f"  {bucket:>8}: {count:>3} classes  {bar}")

### INTERACTIVE: What does this mean for macro F1?

Macro F1 averages per-class F1 scores over all 113 classes, weighted
equally. A class with n=1 val example contributes to the mean with F1
that's either very high or very low — a single prediction flip moves
it drastically.

If 17 classes have n=1, that's **15% of the averaged F1 values** that
are essentially coin flips.

1. If the encoder and decoder macro F1 differ by 0.02, how much of
   that gap could plausibly be noise from the n=1 classes alone?

2. What would you do to decide whether an observed gap is "real"?
   (This is what bootstrap CIs are for — you'll do them in the homework.)

YOUR ANSWERS:

1.

2.


**Connecting to today's lecture:** the 17 n=1 classes are the visible
edge of the `MIN_CLASS_COUNT=5` filter — as covered in the pipeline
module at the start of lecture. A class with 5 total examples loses 1
to val in the stratified split and you land at n=1. That's not a bug,
it's where the filter boundary meets the split.

---
## Part 6: The Bug Hunt (~15 min)

Someone trained a decoder and pushed it to HuggingFace. Here's the repo:
`earino/ecbs5200-week4-flawed-checkpoint`. They trained with the same
base model (Qwen2.5-0.5B), same LoRA config, same data, same 3 epochs.

**Your job:** load it, run it on val, and diagnose what's wrong.
You won't be told the bug. Use the tools you learned this week.
---

In [ ]:
# GUIDED: Load the flawed checkpoint
FLAWED_REPO = "earino/ecbs5200-week4-flawed-checkpoint"
print(f"Loading {FLAWED_REPO}...")
flaw_tokenizer = AutoTokenizer.from_pretrained(FLAWED_REPO)
if flaw_tokenizer.pad_token is None:
    flaw_tokenizer.pad_token = flaw_tokenizer.eos_token

flaw_model = AutoModelForSequenceClassification.from_pretrained(FLAWED_REPO).to(device).eval()
if flaw_model.config.pad_token_id is None:
    flaw_model.config.pad_token_id = flaw_tokenizer.pad_token_id

In [ ]:
# GUIDED: Run inference (reusing the helper from Part 1)
print("Running inference on the flawed checkpoint...")
t0 = time.time()
flaw = run_inference(flaw_model, flaw_tokenizer, val_ds)
print(f"Done in {time.time() - t0:.1f}s")
flaw_acc = (flaw["preds"] == flaw["labels"]).mean()
flaw_f1 = f1_score(flaw["labels"], flaw["preds"], average="macro", zero_division=0)
print(f"  Accuracy: {flaw_acc:.4f}    Macro F1: {flaw_f1:.4f}")
print(f"  Random baseline for 113 classes: {1/113:.4f} accuracy, ~0.003 macro F1")

del flaw_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### INTERACTIVE: First read

The model's accuracy and F1 are near random. What does that tell you,
and — more importantly — what does it NOT tell you?

Several very different failure modes all produce "near random" metrics:
- The model never learned anything (training failed)
- The model collapsed to predicting one class
- The model's output space is scrambled but coherent
- The model is only looking at the wrong features

You need more than aggregate metrics to distinguish these.

YOUR OBSERVATION:



### INTERACTIVE: Build a diagnostic plot

A 113×113 confusion matrix is unreadable as a heatmap at notebook
scale. Here's a different way to see the structure: for each true
class, find the **single prediction it receives most often**. Plot
(true_class_id, most_predicted_class_id) as 113 points.

If the model is random, the points will be scattered. If the model
collapsed to one class, all points will sit on one horizontal line.
If the model's output space is permuted, the points will sit on a
diagonal — but an offset one.

Fill in the plot.

In [ ]:
# INTERACTIVE: Compute the most-predicted class for each true class
flaw_cm = confusion_matrix(flaw["labels"], flaw["preds"],
                           labels=list(range(NUM_LABELS)))
# For each row (true class), the argmax column is what the model most
# commonly predicts when the truth is that class.
most_predicted_per_true = ___   # Shape: (NUM_LABELS,). Hint: .argmax along axis 1.

In [ ]:
# GUIDED: Plot — if the model's output space is intact, points land on y=x
fig, ax = plt.subplots(figsize=(8, 8))
# Only plot rows that have any val examples (no n=0 classes)
has_val = flaw_cm.sum(axis=1) > 0
xs = np.arange(NUM_LABELS)[has_val]
ys = most_predicted_per_true[has_val]
ax.scatter(xs, ys, alpha=0.7, s=25)
ax.plot([0, NUM_LABELS], [0, NUM_LABELS], "k--", alpha=0.3, label="y = x (correct)")
ax.set_xlabel("true class id")
ax.set_ylabel("most-often-predicted class id")
ax.set_title("Diagnostic plot — if the bug is a shift, points sit on a parallel line")
ax.set_xlim(-2, NUM_LABELS + 2)
ax.set_ylim(-2, NUM_LABELS + 2)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

### INTERACTIVE: What do you see?

Look at the scatter. Is it random? Is there a horizontal line? A
diagonal? An OFFSET diagonal?

If the points sit on a line offset from y=x, it suggests **every
prediction is shifted by a constant K relative to the true label**.
If that's what's happening, there's an easy fix: subtract K from all
predictions.

YOUR OBSERVATION:



### INTERACTIVE: Write the fix

Assume you've identified the shift amount K. Given the raw predictions
`flaw["preds"]` (shape `(6430,)`, integers 0-112), what transformation
recovers the intended predictions?

YOUR ANSWER (one-line Python expression):

`fixed_preds = ________________________________________`


In [ ]:
# INTERACTIVE: Apply your fix
# Eyeball the scatter plot to estimate K — find the offset that
# the diagonal sits at.

K = ___   # Your estimate of the shift amount

fixed_preds = (flaw["preds"] - K) % NUM_LABELS
fixed_acc = (fixed_preds == flaw["labels"]).mean()
fixed_f1  = f1_score(flaw["labels"], fixed_preds, average="macro", zero_division=0)

print(f"Before fix: accuracy {flaw_acc:.4f}  macro F1 {flaw_f1:.4f}")
print(f"After  fix: accuracy {fixed_acc:.4f}  macro F1 {fixed_f1:.4f}")
print(f"Recovery:   {(fixed_acc - flaw_acc)*100:+.1f} percentage points")

### The reveal

If you picked the right K, the model's accuracy jumped from ~0.9%
(random) back to ~57% — the same neighborhood as the clean decoder
you saw in Part 1.

<details>
<summary><b>Click after running your fix — what was the bug?</b></summary>

The model wasn't broken. Its **output space was permuted** — during
training, someone shifted the labels by +17 before handing them to the
Trainer, but the val labels stayed canonical. The model trained
perfectly well — on the wrong label frame. At eval time, predictions
pointed at the right CLASS in the training frame, which was the wrong
class in the eval frame.

`fixed_preds = (raw_preds - 17) % 113` undoes the permutation. The
model was fine; the LABELING was wrong.

**Why this is a realistic bug:** in practice, this happens when two
codepaths build `LABEL_LIST` from different sources — alphabetic sort
vs frequency order, insertion order, sentinel-class offsets. One path
tokenizes data for training. Another path evaluates. They disagree on
what label ID 0 means, and nothing catches it because both paths run
clean.

**The lesson:** label ordering must be consistent across EVERY
codepath that touches labels. A single `LABEL_LIST` imported from one
place, used everywhere. If you build it in multiple places, you will
eventually ship this bug to production.

**Week 6 breadcrumb:** next week you'll build a disagreement dataset
— examples where two models agree in different ways. Label hygiene
matters doubly there; if your two models disagree because they were
trained on different label frames, you'll build a useless distillation
target.

</details>

---
## Wrap-up
---

In [ ]:
# GUIDED: Save lab results for homework
results = {
    "encoder": {
        "accuracy":  float((enc["preds"] == enc["labels"]).mean()),
        "macro_f1":  float(f1_score(enc["labels"], enc["preds"],
                                    average="macro", zero_division=0)),
        "ece":       float(enc_ece),
    },
    "decoder": {
        "accuracy":  float((dec["preds"] == dec["labels"]).mean()),
        "macro_f1":  float(f1_score(dec["labels"], dec["preds"],
                                    average="macro", zero_division=0)),
        "ece":       float(dec_ece),
    },
    "tail_tier_breakdown_encoder": br_enc,
    "tail_tier_breakdown_decoder": br_dec,
    "val_count_histogram": buckets,
    "flawed_checkpoint": {
        "accuracy_before_fix": float(flaw_acc),
        "macro_f1_before_fix": float(flaw_f1),
        "shift_k":             int(K),
        "accuracy_after_fix":  float(fixed_acc),
        "macro_f1_after_fix":  float(fixed_f1),
    },
}

with open("week4_lab_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved week4_lab_results.json")
print("Download this file — you'll reference it in the homework.")

# Save the prediction arrays too so homework doesn't re-run inference
np.savez_compressed("week4_lab_predictions.npz",
    encoder_preds=enc["preds"], encoder_labels=enc["labels"],
    encoder_logits=enc["logits"], encoder_max_conf=enc["max_conf"],
    decoder_preds=dec["preds"], decoder_labels=dec["labels"],
    decoder_logits=dec["logits"], decoder_max_conf=dec["max_conf"],
)
print("Saved week4_lab_predictions.npz (~9 MB)")
print("Download this too — homework uses these arrays.")

### What's next (homework)

In the homework you will:

1. **All six slice axes** — add token-length, numeric, opener-I,
   and allcaps to the two axes you did today. Interpret the strongest
   signal axis and the null-result axis.
2. **Temperature scaling** — fit a scalar T on half of val, evaluate
   ECE reduction on the other half, for both models.
3. **Cross-model error analysis** — pull disagreement examples
   (encoder right / decoder wrong, and vice versa), tag them, find
   patterns.
4. **Per-class confusion drill-down** — for the 3 worst classes of
   each model, what classes does it confuse them with?
5. **Bootstrap confidence intervals** — is the encoder-vs-decoder
   macro F1 gap real within your noise floor? Defend with numbers.
6. **Memo (100 points)** — 5 sections, one guiding question each.